# Step 12: Orchestration Patterns

        _Learner Notebook_  
        Estimated time: **90 minutes**  
        Difficulty: **Advanced**

        ## Learning Objectives
        - [ ] Compare sequential, concurrent, and handoff patterns.
- [ ] Use multiple live agents in one orchestration.
- [ ] Measure the tradeoffs at a high level.
- [ ] Choose a pattern based on task shape rather than novelty.

        ## Prerequisites
        - Step 11 completed.


## Table of Contents

1. Setup & Imports
2. Part 1: Introduction
3. Part 2: Core Implementation
4. Part 3: Hands-On Exercises
5. Part 4: Solutions & Discussion
6. Part 5: Best Practices & Tips
7. Summary & Key Takeaways
8. Additional Resources


## Setup & Imports

The next cell adds the repository root to `sys.path` and imports the shared course helpers.
Most notebooks use the same bootstrap so the lesson content can stay focused on the concept,
not on path setup.


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "helpers").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from helpers import (
    LocalTfidfVectorStore,
    SQLiteConversationMemory,
    assert_minimum_python,
    chunk_documents,
    create_agent,
    create_chat_client,
    describe_response,
    load_settings,
    load_text_documents,
    print_banner,
    print_json,
    resolve_notebook_root,
    summarize_session,
    validate_provider_configuration,
)


## Part 1: Introduction

Orchestration patterns are about control flow, not just agent count. The same providers can support very different system designs depending on how tasks move through the team.

Expected output and validation notes:

Expected output snapshot:

- sequential orchestration returns one synthesized answer
- concurrent orchestration returns several perspective-specific answers


## Part 2: Core Implementation

### Sequential and concurrent orchestration


In [ ]:
import asyncio

researcher = create_agent(name="Researcher", instructions="Gather key points on the topic.")
analyst = create_agent(name="Analyst", instructions="Analyze the provided points for implications.")
reporter = create_agent(name="Reporter", instructions="Write a concise summary for engineers.")

async def sequential_orchestration(task: str) -> str:
    research = await researcher.run(task)
    analysis = await analyst.run(research.text)
    report = await reporter.run(analysis.text)
    return report.text

async def concurrent_orchestration(task: str) -> list[str]:
    agents = [
        create_agent(name="MarketLens", instructions="Focus on market implications."),
        create_agent(name="EngineeringLens", instructions="Focus on implementation implications."),
        create_agent(name="ProductLens", instructions="Focus on user and product implications."),
    ]
    responses = await asyncio.gather(*(agent.run(task) for agent in agents))
    return [item.text for item in responses]


## Part 2: Core Implementation

### Simple handoff


In [ ]:
triage = create_agent(
    name="TriageAgent",
    instructions="Classify a request as technical, billing, or general using a single lowercase label.",
)
technical_support = create_agent(name="TechnicalSupport", instructions="Answer technical support questions.")
billing_support = create_agent(name="BillingSupport", instructions="Answer billing support questions.")
general_support = create_agent(name="GeneralSupport", instructions="Answer general questions.")

async def handoff(request: str) -> str:
    label = (await triage.run(f"Classify this request: {request}")).text.lower()
    if "technical" in label:
        return (await technical_support.run(request)).text
    if "billing" in label:
        return (await billing_support.run(request)).text
    return (await general_support.run(request)).text


## Part 3: Hands-On Exercises

Run the same topic through the sequential and concurrent patterns, then compare the output style and latency you observe.

Try the exercise yourself before looking at the solutions in Part 4.


In [ ]:
# TODO: call sequential_orchestration on one topic
# TODO: call concurrent_orchestration on the same topic
# TODO: compare the differences in a short note


## Part 4: Solutions & Discussion

Use this section to compare your implementation with one clear working approach. The goal is not
perfect matching output; the goal is a sound pattern that is easy to explain, debug, and extend.


In [ ]:
sequential_result = await sequential_orchestration("Explain the impact of agent tools on developer productivity.")
concurrent_result = await concurrent_orchestration("Explain the impact of agent tools on developer productivity.")

print_banner("Sequential")
print(sequential_result)
print_banner("Concurrent")
for item in concurrent_result:
    print("-", item)


## Part 5: Best Practices & Tips

        - Use sequential flow when every step depends on the last one.
- Use concurrent flow when views are independent and easy to merge.
- Use handoff when specialization matters more than parallelism.


## Summary & Key Takeaways

You explored the three core orchestration patterns and saw why system shape matters as much as model choice.


## Additional Resources

        - `Step 13 notebook`
- `agent_framework workflow and agent docs`
